In [1]:
import pandas as pd
import glob
import os
import requests
import rasterio
import numpy as np
import json

In [ ]:
import os
import glob
import pandas as pd

cale_dataseturi = os.path.join('..', 'Dataseturi copaci')
fisiere_gasite = glob.glob(os.path.join(cale_dataseturi, "*.csv"))

# Am adăugat mai multe variante posibile pentru Judet
mapare_coloane = {
    'Categorii de fructe': 'Specie_Pomi',
    'Categorii de pomi fructiferi': 'Specie_Pomi',
    'Macroregiuni  regiuni de dezvoltare si judete': 'Judet',
    'Judete': 'Judet',
    'Ani': 'An'
}

lista_df = []

for fisier in fisiere_gasite:
    df = pd.read_csv(fisier)
    
    # 1. CURĂȚARE CRITICĂ: Eliminăm spațiile de la începutul/finalul numelor coloanelor brute
    df.columns = df.columns.str.strip()
    
    # 2. Redenumire
    df.rename(columns=mapare_coloane, inplace=True)
    
    # 3. Mai curățăm o dată după redenumire (just in case)
    df.columns = df.columns.str.strip()
    
    nume = os.path.basename(fisier).lower()
    if 'nr_pomi' in nume:
        df.rename(columns={'Valoare': 'Numar_Pomi'}, inplace=True)
    elif 'kg_pom' in nume:
        df.rename(columns={'Valoare': 'Kg/Pom'}, inplace=True)
    elif 'tone_totale' in nume:
        df.rename(columns={'Valoare': 'Tone_Totale'}, inplace=True)

    print(f"Coloane detectate: {df.columns.tolist()}\n")
    
    # Luăm doar coloanele care chiar au fost găsite/redenumite
    target_cols = ['Specie_Pomi', 'Judet', 'An', 'Numar_Pomi', 'Kg/Pom', 'Tone_Totale']
    cols_prezente = [c for c in target_cols if c in df.columns]
    
    lista_df.append(df[cols_prezente])

if lista_df:
    df_merge = pd.concat(lista_df, ignore_index=True)
    
    # VERIFICARE: Vedem exact ce coloane avem înainte de groupby ca să nu mai dea eroare
    print("Coloane disponibile pentru groupby:", df_merge.columns.tolist())
    
    # Verificăm dacă 'Judet' este în listă, dacă nu, folosim ce găsim
    col_judet = 'Judet' if 'Judet' in df_merge.columns else None
    col_specie = 'Specie_Pomi' if 'Specie_Pomi' in df_merge.columns else None
    col_an = 'An' if 'An' in df_merge.columns else ('Ani' if 'Ani' in df_merge.columns else None)

    # Grupăm doar dacă avem coloanele necesare
    if col_judet and col_specie and col_an:
        df_final = df_merge.groupby([col_specie, col_judet, col_an], as_index=False).first()
        
        print("Succes! Datele au fost consolidate.")
        df_final.to_csv('final_data_copaci.csv', index=False)
    else:
        print("Eroare: Lipsesc coloane de bază (Judet, Specie sau An) pentru a face gruparea!")

Coloane detectate: ['Specie_Pomi', 'Forme de proprietate', 'Judet', 'An', 'UM: Kg/ pom', 'Valoare']

Coloane detectate: ['Specie_Pomi', 'Forme de proprietate', 'Judet', 'An', 'UM: Numar', 'Valoare']

Coloane detectate: ['Specie_Pomi', 'Forme de proprietate', 'Judet', 'An', 'UM: Tone', 'Valoare']

Coloane disponibile pentru groupby: ['Specie_Pomi', 'Judet', 'An', 'Kg/Pom', 'Numar_Pomi', 'Tone_Totale']
Succes! Datele au fost consolidate.


In [3]:
df_master['Max_Kg_Istoric'] = df_master.groupby('Judet')['Kg_per_Pom'].transform('max')

# 2. Modificăm funcția să se uite la acest potențial maxim
def identifica_rootstock_stabil(row):
    potenial_maxim = row['Max_Kg_Istoric']
    
    if potenial_maxim < 20:   # Pragurile pot fi ajustate
        return 1  # Pitic (M9)
    elif potenial_maxim < 38:
        return 2  # Semipitic (M26)
    elif potenial_maxim < 65:
        return 3  # Semiviguros (MM106)
    else:
        return 4  # Viguros (Salbatic)

# 3. Aplicăm pe coloana de potențial
df_master['Rootstock_ID'] = df_master.apply(identifica_rootstock_stabil, axis=1)

print(df_master.head())

           An   Judet Specie_Pomi  Nr_Pomi  Tone_Totale  Kg_per_Pom  \
0   Anul 2000   Bihor       Pruni  1563587        15430          10   
1   Anul 2000   Bihor       Pruni  1563587        15430           7   
2   Anul 2000   Bihor       Pruni  1563587        15430           9   
3   Anul 2000   Bihor       Pruni  1563587        15430           9   
4   Anul 2000   Bihor       Pruni  1563587        15430           8   

   Max_Kg_Istoric  Rootstock_ID  
0              45             3  
1              45             3  
2              45             3  
3              45             3  
4              45             3  


In [4]:
df = pd.read_csv('master_data.csv')
df.head()

target = 'Meri'
df = df[df['Specie_Pomi'] == target]

df.to_csv('copaci_data.csv', index=False)
